In [14]:
import polars as pl
from pathlib import Path

import sys
print(sys.executable)



/Users/mendelgoldberg/.pyenv/versions/3.10.13/bin/python


In [15]:
# Load the Knee Replacement Provider data
file_path = Path("../Lalita/UnderstandingData/merged/Knee Replacement Provider.csv")
df = pl.read_csv(file_path, separator=',', infer_schema_length=10000, ignore_errors=True)

# Make a copy of the original dataframe
df_original = df.clone()
print("Original dataframe copied to df_original.")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")


Original dataframe copied to df_original.
Shape: 139,236 rows × 82 columns


In [16]:

# For columns with exactly 2 unique values where one is 9, recode 9 → 0
cols_to_recode = [
    c for c in df.columns
    if set(df[c].drop_nulls().unique().to_list()) == {9, 0}
    or (
        len(df[c].drop_nulls().unique()) == 2
        and 9 in df[c].drop_nulls().unique().to_list()
    )
]

print(f"Columns to recode (2 unique values, one is 9): {cols_to_recode}")

df = df.with_columns([
    pl.when(pl.col(c) == 9).then(0).otherwise(pl.col(c)).alias(c)
    for c in cols_to_recode
])

print("Done. Value 9 replaced with 0 in the above columns.")


Columns to recode (2 unique values, one is 9): ['Heart Disease', 'High Bp', 'Stroke', 'Circulation', 'Lung Disease', 'Diabetes', 'Kidney Disease', 'Nervous System', 'Liver Disease', 'Cancer', 'Depression', 'Arthritis']
Done. Value 9 replaced with 0 in the above columns.


In [17]:
# Recode 'Pre-Op Q Assisted By': 9 → 0, anything else → 1
df = df.with_columns(
    pl.when(pl.col("Pre-Op Q Assisted By") == 9)
      .then(0)
      .otherwise(1)
      .alias("Pre-Op Q Assisted By")
)

print("'Pre-Op Q Assisted By' recoded: 9 → 0, all other values → 1")
print(df["Pre-Op Q Assisted By"].value_counts())

#Post-Op Q Assisted By 

df = df.with_columns(
    pl.when(pl.col("Post-Op Q Assisted By") == 9)
      .then(0)
      .otherwise(1)
      .alias("Post-Op Q Assisted By")
)

print("'Post-Op Q Assisted By' recoded: 9 → 0, all other values → 1")
print(df["Post-Op Q Assisted By"].value_counts())


'Pre-Op Q Assisted By' recoded: 9 → 0, all other values → 1
shape: (1, 2)
┌──────────────────────┬────────┐
│ Pre-Op Q Assisted By ┆ count  │
│ ---                  ┆ ---    │
│ i32                  ┆ u32    │
╞══════════════════════╪════════╡
│ 1                    ┆ 139236 │
└──────────────────────┴────────┘
'Post-Op Q Assisted By' recoded: 9 → 0, all other values → 1
shape: (2, 2)
┌───────────────────────┬────────┐
│ Post-Op Q Assisted By ┆ count  │
│ ---                   ┆ ---    │
│ i32                   ┆ u32    │
╞═══════════════════════╪════════╡
│ 0                     ┆ 128372 │
│ 1                     ┆ 10864  │
└───────────────────────┴────────┘


In [18]:

# Column types and missing value analysis
# Missing values considered: NaN/null, '', '*', 999

MISSING_SENTINELS_STR = {'*', ''}
MISSING_SENTINELS_NUM = {999, 9}

def count_missing_pl(df: pl.DataFrame, col: str) -> int:
    series = df[col]
    null_count = int(series.is_null().sum())
    if series.dtype in (pl.Utf8, pl.String):
        sentinel_count = int(series.drop_nulls().is_in(list(MISSING_SENTINELS_STR)).sum())
    else:
        sentinel_count = int(series.drop_nulls().is_in(list(MISSING_SENTINELS_NUM)).sum())
    return null_count + sentinel_count

rows, cols_count = df.shape

missing_info = pl.DataFrame({
    'column':        df.columns,
    'dtype':         [str(df[c].dtype) for c in df.columns],
    'total_rows':    [rows] * cols_count,
    'missing_count': [count_missing_pl(df, c) for c in df.columns],
}).with_columns(
    (pl.col('missing_count') / pl.col('total_rows') * 100).round(2).alias('missing_%')
).sort('missing_%', descending=True)

print(f"Dataset shape: {rows:,} rows × {cols_count} columns")

with pl.Config(tbl_rows=-1, tbl_cols=-1, tbl_width_chars=10000):
    print(f"\nColumns with missing values (NaN, '', '*', 999):\n")
    print(missing_info.filter(pl.col('missing_count') > 0))
    print(f"\n--- All columns ---\n")
    print(missing_info)


Dataset shape: 139,236 rows × 82 columns

Columns with missing values (NaN, '', '*', 999):

shape: (61, 5)
┌─────────────────────────────────┬─────────┬────────────┬───────────────┬───────────┐
│ column                          ┆ dtype   ┆ total_rows ┆ missing_count ┆ missing_% │
│ ---                             ┆ ---     ┆ ---        ┆ ---           ┆ ---       │
│ str                             ┆ str     ┆ i64        ┆ i64           ┆ f64       │
╞═════════════════════════════════╪═════════╪════════════╪═══════════════╪═══════════╡
│ Knee Replacement EQ VAS_Post-O… ┆ Float64 ┆ 139236     ┆ 19423         ┆ 13.95     │
│ Post-Op Q Bleeding              ┆ Int64   ┆ 139236     ┆ 17138         ┆ 12.31     │
│ Post-Op Q Urine                 ┆ Int64   ┆ 139236     ┆ 15534         ┆ 11.16     │
│ Post-Op Q Wound                 ┆ Int64   ┆ 139236     ┆ 14740         ┆ 10.59     │
│ Knee Replacement EQ 5D Index P… ┆ Float64 ┆ 139236     ┆ 14322         ┆ 10.29     │
│ Pre-Op Q EQ VAS      

## UMAP Clusters of Missing Values
Builds the missingness matrix, fits UMAP + DBSCAN, and plots the result. Clusters are ordered largest first. Hover on any point to see the top 10 most-missing columns for that cluster. Hover on the centroid marker (larger outlined circle) for a cluster-level summary.

In [19]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "umap-learn"])

  Using cached umap_learn-0.5.11-py3-none-any.whl (90 kB)
  Using cached scikit_learn-1.7.2-cp310-cp310-macosx_10_9_x86_64.whl (9.3 MB)
  Using cached pynndescent-0.6.0-py3-none-any.whl (73 kB)
  Using cached numba-0.64.0-cp310-cp310-macosx_15_0_x86_64.whl
  Using cached llvmlite-0.46.0.tar.gz (193 kB)
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
  Running setup.py clean for llvmlite


  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> [46 lines of output]
      /Users/mendelgoldberg/.pyenv/versions/3.10.13/lib/python3.10/site-packages/wheel/bdist_wheel.py:4: FutureWarning: The 'wheel' package is no longer the canonical location of the 'bdist_wheel' command, and will be removed in a future release. Please update to setuptools v70.1 or later which contains an integrated version of this command.
        warn(
      /Users/mendelgoldberg/.pyenv/versions/3.10.13/lib/python3.10/site-packages/setuptools/_distutils/dist.py:264: UserWarning: Unknown distribution option: 'license_expression'
        warnings.warn(msg)
      running bdist_wheel
      /Users/mendelgoldberg/.pyenv/versions/3.10.13/bin/python /private/var/folders/xk/kg13qzdn74nfkmm57d6yv07m0000gn/T/pip-install-zfnhqe_n/llvmlite_16786e6fd5504362bb216fe0b560bf25/ffi/build.py
      Running: cmake -G Unix Makefiles /private/var/folders/xk/kg13qzdn

Failed to build llvmlite
  Running setup.py install for llvmlite: started
  Running setup.py install for llvmlite: finished with status 'error'


  error: subprocess-exited-with-error
  
  × Running setup.py install for llvmlite did not run successfully.
  │ exit code: 1
  ╰─> [39 lines of output]
      /Users/mendelgoldberg/.pyenv/versions/3.10.13/lib/python3.10/site-packages/wheel/bdist_wheel.py:4: FutureWarning: The 'wheel' package is no longer the canonical location of the 'bdist_wheel' command, and will be removed in a future release. Please update to setuptools v70.1 or later which contains an integrated version of this command.
        warn(
      /Users/mendelgoldberg/.pyenv/versions/3.10.13/lib/python3.10/site-packages/setuptools/_distutils/dist.py:264: UserWarning: Unknown distribution option: 'license_expression'
        warnings.warn(msg)
      running install
      /Users/mendelgoldberg/.pyenv/versions/3.10.13/lib/python3.10/site-packages/setuptools/command/install.py:34: SetuptoolsDeprecationWarning: setup.py install is deprecated. Use build and pip and other standards-based tools.
        warnings.warn(
      runn

CalledProcessError: Command '['/Users/mendelgoldberg/.pyenv/versions/3.10.13/bin/python', '-m', 'pip', 'install', 'umap-learn']' returned non-zero exit status 1.

In [ ]:

# Build missingness matrix, UMAP embedding, and DBSCAN clusters
# Then visualize with interactive scatter — hover on any point shows top missing columns for its cluster
from umap import UMAP
from sklearn.cluster import DBSCAN
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
import pandas as pd
from IPython.display import display, HTML

# Build binary missingness indicator matrix (1 = missing, 0 = present)
def is_missing_np(series: pl.Series) -> np.ndarray:
    if series.dtype in (pl.Utf8, pl.String):
        return (series.is_null() | series.is_in(['', '*'])).to_numpy().astype(np.uint8)
    else:
        return (series.is_null() | series.is_in([999])).to_numpy().astype(np.uint8)

col_names = [c for c in df.columns if c != "has_missing"]
miss_matrix = np.column_stack([is_missing_np(df[c]) for c in col_names])

# Fit UMAP embedding
reducer = UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
embedding = reducer.fit_transform(miss_matrix)

# Cluster UMAP embedding with DBSCAN
dbscan = DBSCAN(eps=0.5, min_samples=10)
cluster_labels = dbscan.fit_predict(embedding)

# Prepare cluster info
cluster_ids = np.unique(cluster_labels)
cluster_sizes = {cid: int((cluster_labels == cid).sum()) for cid in cluster_ids}
cluster_avg_missing = {
    cid: float(miss_matrix[cluster_labels == cid].mean()) * 100
    for cid in cluster_ids
}

# Sort: biggest cluster first, noise always last
def sort_key(cid):
    if cid == -1:
        return (1, 0)
    return (0, -cluster_sizes[cid])

sorted_ids = sorted(cluster_ids.tolist(), key=sort_key)

# Pre-compute top 10 missing columns per cluster (used for hover tooltips)
cluster_top_missing = {}
for cid in cluster_ids:
    mask = cluster_labels == cid
    miss_pcts = miss_matrix[mask].mean(axis=0) * 100
    top_cols = sorted(zip(col_names, miss_pcts), key=lambda x: -x[1])[:10]
    top_str = '<br>'.join([f'  {c}: {p:.1f}%' for c, p in top_cols if p > 0]) or 'none'
    cluster_top_missing[cid] = top_str

palette = px.colors.qualitative.Dark24
cluster_color_map = {}
cluster_name_map = {}
for i, cid in enumerate(sorted_ids):
    avg_miss = cluster_avg_missing[cid]
    size = cluster_sizes[cid]
    if avg_miss == 0:
        color = 'green'
        name = f'Cluster {cid} — no missing (n={size:,})'
    elif cid == -1:
        color = '#aaaaaa'
        name = f'Noise (n={size:,}, avg miss={avg_miss:.1f}%)'
    else:
        color = palette[i % len(palette)]
        name = f'Cluster {cid} (n={size:,}, avg miss={avg_miss:.1f}%)'
    cluster_color_map[cid] = color
    cluster_name_map[cid] = name

# Build DataFrame with top_missing column for hover
plot_df = pd.DataFrame({
    'UMAP1': embedding[:, 0],
    'UMAP2': embedding[:, 1],
    'cluster': cluster_labels,
    'cluster_name': [cluster_name_map[c] for c in cluster_labels],
    'missing_pct': (miss_matrix.sum(axis=1) / miss_matrix.shape[1] * 100).round(1),
    'top_missing': [cluster_top_missing[c] for c in cluster_labels],
})

# Ordered categories so legend follows sort order
plot_df['cluster_name'] = pd.Categorical(
    plot_df['cluster_name'],
    categories=[cluster_name_map[cid] for cid in sorted_ids],
    ordered=True
)

color_discrete_map = {cluster_name_map[cid]: cluster_color_map[cid] for cid in sorted_ids}

# Interactive scatter — click legend to isolate/hide clusters
fig = px.scatter(
    plot_df.sort_values('cluster_name'), x='UMAP1', y='UMAP2',
    color='cluster_name',
    color_discrete_map=color_discrete_map,
    category_orders={'cluster_name': [cluster_name_map[cid] for cid in sorted_ids]},
    custom_data=['cluster_name', 'cluster', 'missing_pct', 'top_missing'],
    labels={'cluster_name': 'Cluster (largest first)', 'missing_pct': 'Missing cols (%)'},
    opacity=0.6
)
fig.update_traces(
    marker=dict(size=4),
    hovertemplate=(
        '<b>%{customdata[0]}</b><br>'
        'Row missing cols: %{customdata[2]}%<br>'
        '<br><b>Top missing columns in cluster:</b><br>'
        '%{customdata[3]}'
        '<extra></extra>'
    )
)

# Add circle markers at each cluster centroid with same summary on hover
for cid in sorted_ids:
    if cid == -1:
        continue
    mask = cluster_labels == cid
    cx = float(embedding[mask, 0].mean())
    cy = float(embedding[mask, 1].mean())
    hover_text = (
        f'<b>Cluster {cid} — Centroid</b><br>'
        f'Size: {cluster_sizes[cid]:,}<br>'
        f'Avg missing: {cluster_avg_missing[cid]:.1f}%<br>'
        f'<br><b>Top missing columns:</b><br>{cluster_top_missing[cid]}'
    )
    fig.add_trace(go.Scatter(
        x=[cx], y=[cy],
        mode='markers',
        marker=dict(symbol='circle', size=10, color=cluster_color_map[cid],
                    line=dict(width=2, color='black')),
        hovertext=hover_text,
        hoverinfo='text',
        showlegend=False,
        name=f'Cluster {cid} centroid'
    ))

fig.update_layout(
    title='UMAP of Missingness Matrix — DBSCAN Clusters (largest first, hover for cluster details)',
    legend_title='Cluster (largest first)',
    legend=dict(itemclick='toggleothers', itemdoubleclick='toggle'),
    width=950, height=680
)

display(HTML(fig.to_html(include_plotlyjs='cdn', full_html=False)))


In [ ]:

# Imputation strategy suggestion for each column with missing values
# Rules:
#   missing > 50%       → consider DROPPING the column
#   string/categorical  → MODE imputation
#   numeric, skewed     → MEDIAN imputation  (|skew| > 1)
#   numeric, symmetric  → MEAN imputation
#   binary (2 values)   → MODE imputation

import scipy.stats as stats

def suggest_imputation(df: pl.DataFrame, col: str, missing_pct: float) -> dict:
    series = df[col].drop_nulls()
    
    # filter out sentinels before profiling
    if series.dtype in (pl.Utf8, pl.String):
        series = series.filter(~series.is_in(['', '*']))
    else:
        series = series.filter(~series.is_in([999]))

    n_unique = series.n_unique()
    dtype = str(df[col].dtype)

    if missing_pct > 50:
        strategy = "⚠️  DROP column (>50% missing)"
        detail = f"{missing_pct:.1f}% missing"
    elif series.dtype in (pl.Utf8, pl.String):
        mode_val = series.mode().to_list()
        strategy = "MODE (categorical)"
        detail = f"mode = {mode_val[0] if mode_val else 'N/A'}"
    elif n_unique == 2:
        mode_val = series.mode().to_list()
        strategy = "MODE (binary)"
        detail = f"mode = {mode_val[0] if mode_val else 'N/A'}"
    else:
        arr = series.to_numpy()
        skewness = stats.skew(arr)
        median_val = float(series.median())
        mean_val   = float(series.mean())
        if abs(skewness) > 1:
            strategy = "MEDIAN (skewed numeric)"
            detail = f"skew={skewness:.2f}, median={median_val:.2f}"
        else:
            strategy = "MEAN (symmetric numeric)"
            detail = f"skew={skewness:.2f}, mean={mean_val:.2f}"

    return {"column": col, "dtype": dtype, "missing_%": round(missing_pct, 2),
            "n_unique": n_unique, "suggestion": strategy, "detail": detail}

# Build suggestion table for columns that have missing values
suggestions = []
for row in missing_info.filter(pl.col("missing_count") > 0).iter_rows(named=True):
    col = row["column"]
    if col == "has_missing":
        continue
    suggestions.append(suggest_imputation(df, col, row["missing_%"]))

suggestion_df = pl.DataFrame(suggestions)

with pl.Config(tbl_rows=-1, tbl_cols=-1, tbl_width_chars=12000, fmt_str_lengths=80):
    print("=== Imputation Strategy Suggestions ===\n")
    print(suggestion_df)


In [ ]:

# Save to parquet in UnderstandingData/output/
output_dir = Path(r"c:\Users\mittall\source\EAISI\NHS\EAISI-NHS\notebooks\Lalita\UnderstandingData\output")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "knee_replacement_providerStep1.parquet"
df.write_parquet(output_path)

print(f"Saved to: {output_path}")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
